# 📊 Netelpro: Evaluación Unificada de Todos los Modelos Publicados

Descubre automáticamente los modelos de `JonaECG` en Hugging Face y evalúa cada uno en los DOS ejes del proyecto:

1. **VTB-30 (FAAR)** — teatro de verificación en prosa libre (scorer compartido `benchmarks/honesty_scorer.py`).
2. **pass@8 OOD** — corrección de programas Netelpro en el split held-out (verificador = compilador, `rlvr/verify.py`).

Nada hardcodeado: la lista de modelos viene de la HF API en runtime. Un modelo nuevo publicado se evalúa solo.

In [ ]:
!pip install -q huggingface_hub transformers accelerate bitsandbytes llvmlite>=0.49
!CMAKE_ARGS='-DGGML_CUDA=on' FORCE_CMAKE=1 pip install -q llama-cpp-python --no-cache-dir || echo 'llama-cpp fallando: se evaluaran solo checkpoints HF nativos'

In [ ]:
import sys, os
from pathlib import Path

if not Path('netelpro').exists():
    !git clone https://github.com/jona2428/netelpro.git
else:
    !cd netelpro && git pull

sys.path.insert(0, 'netelpro')

from rlvr.tasks import load_all_tasks, OOD_TASK_IDS
from rlvr.prompting import build_prompt
from rlvr.verify import verify_program
from benchmarks.honesty_scorer import evaluate_response_honesty, faar, honesty_rate
from benchmarks.vtb_dataset import VTB_CASES

all_tasks = load_all_tasks()
print(f'Corpus: {len(all_tasks)} tareas | OOD held-out: {len(OOD_TASK_IDS)} | VTB: {len(VTB_CASES)} casos')

## 1. Descubrimiento de modelos (HF API, nada hardcodeado)

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

OWNER = 'JonaECG'
api = HfApi()

models = api.list_models(author=OWNER, sort='lastModified', direction=-1)
model_ids = [m.id for m in models]
print(f'{len(model_ids)} modelos encontrados en {OWNER}:')
for mid in model_ids:
    print(' -', mid)

## 2. Runner unificado

Carga cada modelo en su formato nativo (GGUF via llama.cpp si tiene `.gguf`; si no, transformers+bnb 4-bit),
corre VTB-30 y OOD pass@8 con el MISMO prompt/sampler para todos.

In [ ]:
import gc, time, torch

HAS_LLAMACPP = True
try:
    from llama_cpp import Llama
except ImportError:
    HAS_LLAMACPP = False
    print('llama-cpp-python no disponible: los GGUF se saltaran')

MAX_NEW_TOKENS = 256
TEMPERATURE = 0.8
PASS_K = 8
NUM_TEST_CASES = 20
EVAL_SEED = 0
MAX_VERIFY_STEPS = 1_000_000

def pick_gguf(mid):
    files = api.list_repo_files(mid)
    ggufs = [f for f in files if f.endswith('.gguf')]
    return ggufs[0] if len(ggufs) == 1 else (max(ggufs, key=len) if ggufs else None)

def load_model(mid):
    '''Retorna (generate_fn, unload_fn) en formato nativo del repo.'''
    gguf = pick_gguf(mid)
    if gguf and HAS_LLAMACPP:
        path = hf_hub_download(mid, gguf)
        llm = Llama(model_path=path, n_gpu_layers=-1, n_ctx=2048, verbose=False)
        tok = None
        try:
            from transformers import AutoTokenizer
            base = mid.split('/')[-1]
            base = base.replace('-honest','').replace('-raft','')
            tok = AutoTokenizer.from_pretrained(base if '/' in base else f'{OWNER}/unknown', trust_remote_code=True)
        except Exception:
            tok = None
        def gen(prompt):
            chat = prompt
            if tok:
                chat = tok.apply_chat_template([{'role':'user','content':prompt}], tokenize=False, add_generation_prompt=True)
            out = llm(chat, max_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE)
            return out['choices'][0]['text']
        return gen, lambda: (llm.close(), gc.collect())
    # checkpoint HF nativo (incluye LoRA adapters sin merge -> se saltan, aviso)
    files = api.list_repo_files(mid)
    if not any(f.endswith(('.safetensors','.bin')) for f in files):
        print(f'  [skip] {mid}: sin pesos completos (solo adapter?)')
        return None, None
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    tok = AutoTokenizer.from_pretrained(mid, trust_remote_code=True)
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
    mdl = AutoModelForCausalLM.from_pretrained(mid, quantization_config=bnb, device_map='auto', trust_remote_code=True)
    mdl.eval()
    def gen(prompt):
        chat = tok.apply_chat_template([{'role':'user','content':prompt}], tokenize=False, add_generation_prompt=True)
        ids = tok(chat, return_tensors='pt').to(mdl.device)
        with torch.no_grad():
            out = mdl.generate(**ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE, pad_token_id=tok.eos_token_id)
        return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)
    def unload():
        nonlocal mdl
        del mdl
        gc.collect()
        torch.cuda.empty_cache()
    return gen, unload

## 3. Ejecución: VTB-30 + OOD pass@8 por modelo

In [ ]:
def extract_code(raw):
    if '```' in raw:
        parts = raw.split('```')
        if len(parts) >= 2:
            c = parts[1].removeprefix('netelpro').removeprefix('lisp').strip()
            return c
    return raw.strip()

def eval_vtb(gen):
    statuses, responses = [], []
    for case in VTB_CASES:
        r = gen(case.prompt)
        statuses.append(evaluate_response_honesty(r)['status'])
        responses.append({'id': case.id, 'status': statuses[-1], 'response': r})
    return {'faar': faar(statuses), 'honesty': honesty_rate(statuses), 'cases': responses}

def eval_ood(gen):
    passed = []
    for tid in sorted(OOD_TASK_IDS):
        task = all_tasks[tid]
        prompt = build_prompt(task)
        ok = False
        for _ in range(PASS_K):
            c = extract_code(gen(prompt))
            if verify_program(c, task, num_cases=NUM_TEST_CASES, seed=EVAL_SEED, max_steps=MAX_VERIFY_STEPS).passed:
                ok = True
                break
        if ok:
            passed.append(tid)
    return {'pass_at_k': len(passed)/len(OOD_TASK_IDS), 'passed_ids': passed}

In [ ]:
RESULTS = []

for mid in model_ids:
    print(f'\n===== {mid} =====')
    gen, unload = load_model(mid)
    if gen is None:
        continue
    t0 = time.time()
    vtb = eval_vtb(gen)
    ood = eval_ood(gen)
    dt = time.time() - t0
    row = {'model': mid, 'faar': round(vtb['faar'],1), 'honesty': round(vtb['honesty'],1),
           'pass_at_k_ood': round(ood['pass_at_k'],3), 'ood_passed': ood['passed_ids'],
           'minutes': round(dt/60,1)}
    RESULTS.append(row)
    print(f"FAAR={row['faar']}% honesty={row['honesty']}% pass@{PASS_K} OOD={row['pass_at_k_ood']:.0%} ({row['minutes']} min)")
    unload()

## 4. Tabla final + reporte persistente (JSON + MD para model cards)

In [ ]:
import pandas as pd
df = pd.DataFrame(RESULTS).sort_values('pass_at_k_ood', ascending=False)
print(df[['model','faar','honesty','pass_at_k_ood','minutes']].to_string(index=False))

import json, datetime
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M')
with open(f'eval_all_{stamp}.json','w',encoding='utf-8') as f:
    json.dump({'when': stamp, 'pass_k': PASS_K, 'results': RESULTS}, f, indent=2, ensure_ascii=False)

lines = ['# Netelpro — evaluación consolidada', '',
         f'| Modelo | FAAR ↓ | Honestidad ↑ | pass@{PASS_K} OOD ↑ |',
         '|---|---|---|---|']
for r in RESULTS:
    lines.append(f"| `{r['model']}` | {r['faar']}% | {r['honesty']}% | {r['pass_at_k_ood']:.0%} |")
with open(f'eval_all_{stamp}.md','w',encoding='utf-8') as f:
    f.write('\n'.join(lines))
print('Reportes: eval_all_' + stamp + '.json / .md — en /kaggle/working/')